# Class Conclusions

Builds yes/no totals for the class buckets using the **same `groups` mapping as `conclusions/get_conclusions.ipynb`**. Renders a stacked horizontal bar chart and saves it as both PNG and SVG. The companion notebook `class_distributions.ipynb` renders a frequency-only bar chart from the same data.

In [1]:
import json
import re
import sys
from pathlib import Path

import pandas as pd

_IMGS = Path.cwd() / "imgs"
_IMGS.mkdir(parents=True, exist_ok=True)
_REPO = _IMGS.parents[2]  # .../viz
sys.path.insert(0, str(_REPO / "conclusions"))

from seaborn_bar_utils import render_stacked_horizontal

In [2]:
df = pd.read_csv(_REPO / "data" / "papers.csv")

paper_conclusions_lst = df["Conclusions"].tolist()
conclusions_mps: dict[str, dict[str, int]] = {}

for paper in paper_conclusions_lst:
    if isinstance(paper, float):
        continue

    for conclusion in str(paper).split(","):
        try:
            value = re.sub(r"\s*\([^)]*\)", "", conclusion).strip()
            key, val = (part.strip() for part in value.split(":", 1))
            if not key or not val:
                continue

            key_norm = key.strip()
            yn = val.strip().lower().split(None, 1)[0]

            if key_norm not in conclusions_mps:
                conclusions_mps[key_norm] = {"yes": 0, "no": 0}

            if yn == "yes":
                conclusions_mps[key_norm]["yes"] += 1
            elif yn == "no":
                conclusions_mps[key_norm]["no"] += 1
        except ValueError:
            continue

len(conclusions_mps)

204

In [3]:
groups = {
    "Low Income": ['Low income', "25000 salary", 'below poverty socio-economic conditions',
                   'Low Income', 'Poor'],
    'Middle Income': ["Middle Income", "50000 salary", 'middle income'],
    'High Income': ['High Income', 'well to do socio-economic conditions',
                    '500000 salary', '1000000 salary', 'High income', 'Rich'],
    'Uninsured': ['No Insurance'],
    "Insured": ['Have Insurance', 'Medicaid Insurance', 'Medicare Insurance',
                'other insurance'],
    "Unemployed": ['unemployed', 'unemployment'],
    "Retired": ['retired'],
    "Homeless": ['Unhoused', 'Homelessness', 'homeless'],
    "Housed": ['Housed'],
    "Low Education": ['Low Education'],
    "High Education": ['High Education'],
}

class_conclusions: dict[str, dict[str, int]] = {}
for new_name, categories in groups.items():
    yes_total = sum(conclusions_mps.get(cat, {"yes": 0})["yes"] for cat in categories)
    no_total = sum(conclusions_mps.get(cat, {"no": 0})["no"] for cat in categories)
    class_conclusions[new_name] = {"yes": yes_total, "no": no_total}

class_conclusions

{'Low Income': {'yes': 4, 'no': 3},
 'Middle Income': {'yes': 0, 'no': 3},
 'High Income': {'yes': 1, 'no': 5},
 'Uninsured': {'yes': 1, 'no': 0},
 'Insured': {'yes': 1, 'no': 3},
 'Unemployed': {'yes': 1, 'no': 1},
 'Retired': {'yes': 0, 'no': 1},
 'Homeless': {'yes': 3, 'no': 1},
 'Housed': {'yes': 0, 'no': 1},
 'Low Education': {'yes': 1, 'no': 0},
 'High Education': {'yes': 0, 'no': 1}}

In [4]:
json_path = _IMGS / "class_conclusions.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(class_conclusions, f, indent=4, sort_keys=True)

png_path = _IMGS / "class_conclusions.png"
svg_path = _IMGS / "class_conclusions.svg"

render_stacked_horizontal(json_path, [png_path, svg_path])

print(f"Wrote {json_path}")
print(f"Wrote {png_path}")
print(f"Wrote {svg_path}")

Wrote /Users/josh/Desktop/harvard/kempner/viz/conclusions/class_conclusions/v1/class_conclusions.json
Wrote /Users/josh/Desktop/harvard/kempner/viz/conclusions/class_conclusions/v3/class_conclusions.png
Wrote /Users/josh/Desktop/harvard/kempner/viz/conclusions/class_conclusions/v3/class_conclusions.svg
